# DC1 – Emulating Global Ocean: Evaluation Quickstart

This notebook demonstrates the complete DC1 evaluation workflow:

1. **Generate** a sample submission (random noise, surface only)
2. **Inspect** the dataset structure
3. **Validate** the format against the DC1 specification
4. **Run** the evaluation pipeline
5. **Visualise** the results

> **DC1 is a surface-only challenge.** Predictions cover the global ocean at
> 0.25° resolution and include 10 daily lead times per forecast initialisation date.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-ai-data-challenges/dc1-emulating-global-ocean/blob/main/notebooks/evaluation_quickstart.ipynb)


## 0. Setup

If running on **Colab**, install the project first:

```bash
!pip install git+https://github.com/ocean-ai-data-challenges/dc1-emulating-global-ocean.git
```

Otherwise, ensure the `dc-env` conda environment is activated  
(see the [Quickstart guide](https://dc1-emulating-global-ocean.readthedocs.io/en/latest/content/quickstart.html)).


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Engine 'argo' loading failed", category=RuntimeWarning)

import json
from pathlib import Path

import numpy as np
import xarray as xr

# Ensure the project root is on the Python path (works from notebooks/)
import sys
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 1. Generate a sample submission

DC1 predictions are **surface-only** 2-D fields on a global 0.25° grid.
Each forecast file covers 10 daily lead times starting from an initialisation date.

The cell below generates a minimal random-noise dataset that conforms to the
DC1 specification — useful to test the full pipeline before plugging in your
own model.


In [ ]:
import pandas as pd

# ── DC1 grid specification ────────────────────────────────────────────
DC1_LAT       = np.arange(-78.0, 90.25, 0.25)
DC1_LON       = np.arange(-180.0, 180.0, 0.25)
DC1_DEPTH     = np.array([0.494025])          # surface only
DC1_VARIABLES = ["zos", "thetao", "so", "uo", "vo"]
N_LEAD_TIMES  = 10   # 0 … 9 days

OUTPUT_DIR = Path("/tmp/dc1_quickstart_sample")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

# Generate one forecast file per week over 4 weeks (enough to test the pipeline)
init_dates = pd.date_range("2024-01-01", periods=4, freq="7D")

for init_date in init_dates:
    lead_times = np.arange(N_LEAD_TIMES)
    ds = xr.Dataset(
        {
            var: xr.DataArray(
                rng.standard_normal((N_LEAD_TIMES, len(DC1_LAT), len(DC1_LON))).astype("float32"),
                dims=["time", "lat", "lon"],
            )
            for var in DC1_VARIABLES
        },
        coords={
            "time":  lead_times,
            "lat":   DC1_LAT,
            "lon":   DC1_LON,
        },
    )
    store_path = OUTPUT_DIR / f"{init_date:%Y%m%d}.zarr"
    ds.to_zarr(str(store_path), mode="w", consolidated=True)

zarr_files = sorted(OUTPUT_DIR.glob("*.zarr"))
print(f"Generated {len(zarr_files)} forecast files:")
for f in zarr_files:
    print(f"  {f.name}")


## 2. Inspect the dataset

Open one of the generated files and verify its structure.


In [ ]:
ds = xr.open_zarr(str(zarr_files[0]))
print(f"Latitude:   {ds.lat.values[0]:.2f} to {ds.lat.values[-1]:.2f}  ({len(ds.lat)} pts, step {np.diff(ds.lat.values[:2])[0]:.2f}°)")
print(f"Longitude:  {ds.lon.values[0]:.2f} to {ds.lon.values[-1]:.2f}  ({len(ds.lon)} pts, step {np.diff(ds.lon.values[:2])[0]:.2f}°)")
print(f"Time:       {ds.time.values[0]} to {ds.time.values[-1]}  ({len(ds.time)} lead times)")
print(f"Variables:  {list(ds.data_vars)}")
ds


## 3. Validate the submission

Before running the full evaluation, check that your predictions conform to
the DC1 grid specification (dimensions, variables, NaN fraction).


In [ ]:
import subprocess

result = subprocess.run(
    [
        sys.executable, "-m", "dc1.submit", "validate",
        str(OUTPUT_DIR),
        "--model-name", "QuickstartSample",
        "--quick",
    ],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


## 4. Inspect the DC1 specification

The `info` subcommand shows the full evaluation configuration (grid, variables,
reference datasets, evaluation period).


In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "dc1.submit", "info", "--config", "dc1"],
    capture_output=True, text=True,
    cwd=str(PROJECT_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


## 5. Visualise existing results (GloNet baseline)

If a full evaluation has already been run, load and visualise the scores.
DC1 evaluates surface predictions against five reference datasets:
**Argo profiles**, **GLORYS12**, **Jason-3**, **SARAL**, and **SWOT**.


In [ ]:
import matplotlib.pyplot as plt

# Try dc1_output/results first, then fall back to leaderboard_results
results_path = PROJECT_ROOT / "dc1_output" / "results" / "results_glonet.json"
if not results_path.exists():
    results_path = PROJECT_ROOT / "dc1" / "leaderboard_results" / "results_glonet.json"

if results_path.exists():
    with open(results_path) as f:
        data = json.load(f)
    model_name = data["dataset"]
    records = data["results"][model_name]
    print(f"Loaded results for model: {model_name}  ({len(records)} evaluation records)")
    print(f"Reference datasets: {sorted(set(r['ref_alias'] for r in records))}")
else:
    print("No results file found. Run the full evaluation first:")
    print(f"  python -m dc1.submit run {OUTPUT_DIR} --model-name QuickstartSample --data-directory ./dc1_output")
    records = []
    model_name = "QuickstartSample"


In [ ]:
# Parse RMSE scores grouped by reference dataset and variable
scores = {}  # {ref_alias: {variable: {lead_time: value}}}

for record in records:
    ref = record["ref_alias"]
    lt  = record.get("lead_time")
    for metric in record["result"]:
        if metric["Metric"] not in ("rmse", "rmsd"):
            continue
        var = metric["Variable"]
        scores.setdefault(ref, {}).setdefault(var, {})[lt] = metric["Value"]

# RMSE vs lead time for GLORYS reference
if "glorys" in scores:
    glorys = scores["glorys"]
    fig, ax = plt.subplots(figsize=(10, 5))
    for var_name in sorted(glorys):
        lts  = sorted(glorys[var_name].keys())
        vals = [glorys[var_name][lt] for lt in lts]
        ax.plot(lts, vals, marker="o", label=var_name)
    ax.set_xlabel("Lead time (days)")
    ax.set_ylabel("RMSE")
    ax.set_title(f"{model_name} – Surface RMSE vs GLORYS12")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No GLORYS results to plot. Run the evaluation first.")


In [ ]:
# Compare RMSE across all reference datasets (SSH variable)
ssh_vars = [v for ref in scores for v in scores[ref] if "ssh" in v.lower() or "height" in v.lower()]
ssh_vars = sorted(set(ssh_vars))

if ssh_vars and len(scores) > 1:
    fig, axes = plt.subplots(1, len(ssh_vars), figsize=(5 * len(ssh_vars), 4), squeeze=False)
    for ax, var_name in zip(axes[0], ssh_vars):
        for ref in sorted(scores):
            if var_name in scores[ref]:
                lts  = sorted(scores[ref][var_name].keys())
                vals = [scores[ref][var_name][lt] for lt in lts]
                ax.plot(lts, vals, marker=".", label=ref)
        ax.set_xlabel("Lead time (days)")
        ax.set_ylabel("RMSE")
        ax.set_title(var_name)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"{model_name} – SSH RMSE by reference dataset", y=1.02)
    plt.tight_layout()
    plt.show()
elif not scores:
    print("No scores available. Run the evaluation first.")


## Next steps

- Replace the random-noise sample with your model's surface predictions
- Run the full evaluation:
  ```bash
  python -m dc1.submit run /path/to/your/model --model-name MyModel --data-directory ./dc1_output
  ```
- Submit results for the [leaderboard](https://dc1-emulating-global-ocean.readthedocs.io/en/latest/content/submissions.html#participating-in-the-public-leaderboard)
- See the [full submission workflow](submit.ipynb) notebook for a step-by-step guide

See the [full documentation](https://dc1-emulating-global-ocean.readthedocs.io/) for details.
